# Notebook 8 — Feature Scaling
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
print(f"Dataset loaded: {df.shape[0]:,} rows")
print(df[numeric_cols].describe().loc[['min','max','mean','std']].round(2))


Dataset loaded: 7,043 rows
      tenure  MonthlyCharges  TotalCharges
min     0.00           18.25          0.00
max    72.00          118.75       8684.80
mean   32.37           64.76       2279.73
std    24.56           30.09       2266.79


---
## 1. Why Feature Scaling?

### Understand
Many algorithms compute distances or gradients directly from raw feature values — if
features live on wildly different scales, the larger-magnitude feature dominates purely
because of its units, not its actual importance.

### Demonstrate
**Concrete example from this dataset:** `TotalCharges` ranges roughly 0-8,684 while
`SeniorCitizen` is only 0 or 1. Without scaling, a distance-based algorithm would treat a
$1 difference in `TotalCharges` as equal in importance to a full category change in
`SeniorCitizen` — clearly wrong.


---
## 2. Scale-Sensitive vs 3. Scale-Insensitive Algorithms

### Understand
**Scale-sensitive:** KNN, SVM, logistic/linear regression (via gradient descent), PCA,
neural networks — all directly affected by feature magnitude.
**Scale-insensitive:** tree-based models (Decision Trees, Random Forest, Gradient
Boosting) — they split on thresholds per feature independently, so relative scale
doesn't matter.

### Implement


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X = df[numeric_cols]
y = (df['Churn'] == 'Yes').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Unscaled performance
lr_unscaled = LogisticRegression(max_iter=1000).fit(X_train, y_train)
rf_unscaled = RandomForestClassifier(random_state=42).fit(X_train, y_train)
print(f"LogisticRegression (scale-sensitive) accuracy, UNSCALED data: {lr_unscaled.score(X_test, y_test):.4f}")
print(f"RandomForest (scale-insensitive) accuracy, UNSCALED data    : {rf_unscaled.score(X_test, y_test):.4f}")


LogisticRegression (scale-sensitive) accuracy, UNSCALED data: 0.7729
RandomForest (scale-insensitive) accuracy, UNSCALED data    : 0.7523


**Finding:** Both models still ran (this dataset's scale differences aren't extreme
enough to break `LogisticRegression`), but scaling will still change the logistic
regression's coefficients and convergence behavior — the comparison below (Topic 8) makes
the difference concrete.


---
## 4. Normalization vs 5. Standardization vs 6. Min-Max Scaling

### Understand
- **Standardization** (`StandardScaler`): rescales to mean=0, std=1. Doesn't bound values
  to a fixed range.
- **Min-Max Scaling** (`MinMaxScaler`): rescales to a fixed range, usually [0, 1].
  "Normalization" is often used interchangeably with min-max scaling.
- Both preserve the *shape* of the distribution — neither fixes skew (that's Notebook 9's
  job).

### Implement


In [3]:
standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

standardized = pd.DataFrame(standard_scaler.fit_transform(df[numeric_cols]), columns=numeric_cols)
minmax_scaled = pd.DataFrame(minmax_scaler.fit_transform(df[numeric_cols]), columns=numeric_cols)

print("StandardScaler result (mean~0, std~1):")
print(standardized.describe().loc[['mean','std']].round(3))
print("\nMinMaxScaler result (range exactly [0,1]):")
print(minmax_scaled.describe().loc[['min','max']].round(3))


StandardScaler result (mean~0, std~1):
      tenure  MonthlyCharges  TotalCharges
mean    -0.0            -0.0          -0.0
std      1.0             1.0           1.0

MinMaxScaler result (range exactly [0,1]):
     tenure  MonthlyCharges  TotalCharges
min     0.0             0.0           0.0
max     1.0             1.0           1.0


---
## 7. `StandardScaler`, `MinMaxScaler`, `RobustScaler`, `MaxAbsScaler` — Full Comparison

### Understand
- **`StandardScaler`**: mean=0, std=1. Sensitive to outliers (mean/std are themselves
  outlier-sensitive).
- **`MinMaxScaler`**: fixed [0,1] range. Very sensitive to outliers (a single extreme
  value stretches the whole range).
- **`RobustScaler`**: uses median and IQR instead of mean/std — resistant to outliers.
- **`MaxAbsScaler`**: scales by the maximum absolute value, preserving sparsity (0 stays
  0) — most relevant for already-sparse data.

### Implement


In [4]:
robust_scaler = RobustScaler()
maxabs_scaler = MaxAbsScaler()

robust_scaled = pd.DataFrame(robust_scaler.fit_transform(df[numeric_cols]), columns=numeric_cols)
maxabs_scaled = pd.DataFrame(maxabs_scaler.fit_transform(df[numeric_cols]), columns=numeric_cols)

comparison = pd.DataFrame({
    'Original (TotalCharges)': df['TotalCharges'].describe(),
    'StandardScaler': standardized['TotalCharges'].describe(),
    'MinMaxScaler': minmax_scaled['TotalCharges'].describe(),
    'RobustScaler': robust_scaled['TotalCharges'].describe(),
    'MaxAbsScaler': maxabs_scaled['TotalCharges'].describe(),
}).round(3)
print(comparison.loc[['min','max','mean','std']])


      Original (TotalCharges)  StandardScaler  MinMaxScaler  RobustScaler  \
min                     0.000          -1.006         0.000        -0.412   
max                  8684.800           2.826         1.000         2.152   
mean                 2279.734          -0.000         0.262         0.261   
std                  2266.794           1.000         0.261         0.669   

      MaxAbsScaler  
min          0.000  
max          1.000  
mean         0.262  
std          0.261  


---
## 8. How Outliers Affect Scaling

### Understand
Since this dataset has zero true statistical outliers (Notebook 6), all four scalers
behave similarly here. To make the outlier-sensitivity difference concrete, this section
deliberately injects one synthetic extreme value and re-compares.

### Implement


In [5]:
demo = df[numeric_cols].copy()
demo.loc[demo.index[0], 'TotalCharges'] = 500_000   # one deliberate, extreme synthetic outlier

standard_demo = StandardScaler().fit_transform(demo)[:, 2]
minmax_demo = MinMaxScaler().fit_transform(demo)[:, 2]
robust_demo = RobustScaler().fit_transform(demo)[:, 2]

print("Effect of ONE extreme outlier on the scaled range of the OTHER 7,042 normal values:")
print(f"  StandardScaler -> normal-value range: [{standard_demo[1:].min():.3f}, {standard_demo[1:].max():.3f}]")
print(f"  MinMaxScaler   -> normal-value range: [{minmax_demo[1:].min():.3f}, {minmax_demo[1:].max():.3f}]  <- crushed near 0")
print(f"  RobustScaler   -> normal-value range: [{robust_demo[1:].min():.3f}, {robust_demo[1:].max():.3f}]  <- barely affected")


Effect of ONE extreme outlier on the scaled range of the OTHER 7,042 normal values:
  StandardScaler -> normal-value range: [-0.370, 0.998]
  MinMaxScaler   -> normal-value range: [0.000, 0.017]  <- crushed near 0
  RobustScaler   -> normal-value range: [-0.411, 2.149]  <- barely affected


**Finding:** `MinMaxScaler` is dramatically distorted by a single extreme value —
every normal customer gets crushed into a tiny sliver near 0, since the scaler's [0,1]
range is now defined by the one outlier's magnitude. `RobustScaler` is barely affected,
since it's built from the median/IQR, not the min/max. **This is why `RobustScaler` is
the safer default choice whenever a dataset's outlier status is uncertain** — though for
THIS dataset's actual (outlier-free) numeric columns, `StandardScaler` is equally valid
and more standard for algorithms like logistic regression.


---
## Technique Selected for This Dataset

### Documentation (Problem / Analysis / Technique / Reason / Implementation / Result / Impact)
- **Problem:** `tenure`, `MonthlyCharges`, `TotalCharges` live on very different scales
  (0-72, 18-119, 0-8,684).
- **Analysis:** All three are confirmed outlier-free (Notebook 6), so outlier-robustness
  isn't the deciding factor here.
- **Technique Selected:** `StandardScaler`.
- **Reason:** The standard, well-understood default for scale-sensitive linear models
  when outliers aren't a concern — `RobustScaler` would also be defensible, but offers no
  extra benefit here since there's nothing to be robust against.
- **Implementation:** below.
- **Result:** All three numeric features now share a comparable, mean-centered scale.
- **Impact:** A distance- or gradient-based model can now weigh these three features by
  their actual relationship with the target, not by which one happens to have the largest
  raw numbers.


In [6]:
final_scaler = StandardScaler()
df[[f'{c}_scaled' for c in numeric_cols]] = final_scaler.fit_transform(df[numeric_cols])
print(df[[f'{c}_scaled' for c in numeric_cols]].describe().loc[['mean','std']].round(3))


      tenure_scaled  MonthlyCharges_scaled  TotalCharges_scaled
mean           -0.0                   -0.0                 -0.0
std             1.0                    1.0                  1.0


---
## Summary

| Scaler | Best For | Outlier Sensitivity |
|---|---|---|
| StandardScaler | General default for scale-sensitive models | Moderate (mean/std shift with outliers) |
| MinMaxScaler | Fixed-range needs (e.g., neural net inputs) | High — one outlier can crush the whole range |
| RobustScaler | Data with known/suspected outliers | Low — built from median/IQR |
| MaxAbsScaler | Sparse data, preserving zero | Moderate |

**Decision for this dataset: `StandardScaler`**, justified by the confirmed absence of
outliers (Notebook 6) rather than a default guess.

**Next notebook:** `09_Data_Transformation.ipynb` — addressing `TotalCharges`'s skew,
which scaling alone does not fix.
